# Stock Backtesting Notebook

This notebook performs comprehensive backtesting of stock forecasting models with quarterly validation windows.

## Features
- **Quarterly Validation**: Backtesting windows end at the end of each quarter
- **Multi-Stock Support**: Backtests all configured stocks
- **Hyperparameter Integration**: Uses optimized hyperparameters from forecasting notebook
- **Performance Metrics**: MAPE, RMSE, R², and other key metrics
- **Visualization**: Comprehensive charts showing backtesting results
- **Results Export**: Saves backtesting results for analysis


## 1. Setup and Configuration


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import our modules
from src.data_preprocess.stock_data_loader import StockDataLoader
from src.forecasting import WeeklyAggregator, DynamicFeatureEngineer, StandaloneBacktester

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Setup complete!")


In [ ]:
# Configuration
STOCK_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'NVDA']  # Stocks to backtest
FORECAST_HORIZON = 4  # 4 weeks ahead
MIN_TRAINING_QUARTERS = 6  # Minimum 1.5 years of training data
TARGET_COLUMN = 'Close'

# Quarterly backtesting parameters
QUARTERLY_PARAMS = {
    'initial_train_size': 13,  # 1 year of quarterly data (52 weeks / 4)
    'test_size': 1,  # 1 quarter ahead
    'step_size': 1,  # Move forward 1 quarter each iteration
    'min_train_size': 6,  # Minimum 1.5 years of training data
    'target_column': TARGET_COLUMN,
    'forecast_horizon': FORECAST_HORIZON
}

# Hyperparameters from stock forecasting notebook (optimized)
OPTIMIZED_HYPERPARAMS = {
    'n_estimators': 200,
    'max_depth': 5,
    'learning_rate': 0.1,
    'num_leaves': 31,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1
}

print(f"🎯 Backtesting Configuration:")
print(f"   • Target Stocks: {', '.join(STOCK_SYMBOLS)}")
print(f"   • Forecast Horizon: {FORECAST_HORIZON} weeks")
print(f"   • Validation Windows: End of each quarter")
print(f"   • Minimum Training: {MIN_TRAINING_QUARTERS} quarters")
print(f"   • Using optimized hyperparameters from forecasting notebook")


## 2. Data Loading and Preparation


In [ ]:
# Load data for all stocks
print(f"📥 Loading data for {len(STOCK_SYMBOLS)} stocks...")
data_loader = StockDataLoader()

try:
    # Load all stocks at once
    stock_data = data_loader.load_saved_data(STOCK_SYMBOLS)
    
    print(f"✅ Successfully loaded data for {len(stock_data)} stocks")
    
    # Display summary for each stock
    print(f"\n📊 Data Summary by Stock:")
    for symbol in STOCK_SYMBOLS:
        if symbol in stock_data:
            data = stock_data[symbol]
            print(f"   • {symbol}: {len(data)} days, "
                  f"${data['Close'].mean():.2f} avg price, "
                  f"{data['Volume'].mean():,.0f} avg volume")
        else:
            print(f"   • {symbol}: ❌ No data found")
    
    # Check date ranges
    print(f"\n📅 Date Ranges:")
    for symbol in STOCK_SYMBOLS:
        if symbol in stock_data:
            data = stock_data[symbol]
            if 'Date' in data.columns:
                data['Date'] = pd.to_datetime(data['Date'], utc=True).dt.tz_localize(None)
                print(f"   • {symbol}: {data['Date'].min().strftime('%Y-%m-%d')} to {data['Date'].max().strftime('%Y-%m-%d')}")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    raise


## 3. Weekly Data Aggregation and Feature Engineering


In [ ]:
# Convert daily data to weekly frequency and create features
print("📊 Converting to weekly data and creating features...")

weekly_aggregator = WeeklyAggregator(
    price_columns=['Open', 'High', 'Low', 'Close'],
    volume_columns=['Volume']
)

# Simple feature engineering functions (from forecasting notebook)
def create_simple_features(data):
    """Create simple features for backtesting."""
    features = data.copy()
    
    # Simple lag features
    for lag in [1, 2, 4]:
        features[f'close_lag_{lag}'] = features['Close'].shift(lag)
    
    # Simple rolling features
    for window in [4, 8]:
        features[f'close_ma_{window}'] = features['Close'].rolling(window=window).mean()
        features[f'close_std_{window}'] = features['Close'].rolling(window=window).std()
    
    # Price change features
    features['price_change_1w'] = features['Close'].pct_change(1)
    features['price_change_4w'] = features['Close'].pct_change(4)
    
    # Volume features
    features['volume_ma_4'] = features['Volume'].rolling(window=4).mean()
    features['volume_ratio'] = features['Volume'] / features['volume_ma_4']
    
    # Time features
    features['week_of_year'] = features.index.isocalendar().week
    features['month'] = features.index.month
    features['quarter'] = features.index.quarter
    
    return features

def create_simple_targets(data, horizon=4):
    """Create target variables for backtesting."""
    targets = data.copy()
    
    # Create target: percentage change in price over horizon weeks
    targets[f'target_{horizon}w_pct'] = (targets['Close'].shift(-horizon) - targets['Close']) / targets['Close'] * 100
    
    return targets

# Process each stock
backtesting_data = {}
for symbol in STOCK_SYMBOLS:
    if symbol in stock_data:
        print(f"   Processing {symbol}...")
        try:
            daily_data = stock_data[symbol].copy()
            
            # Convert Date column to datetime and set as index
            if 'Date' in daily_data.columns:
                daily_data['Date'] = pd.to_datetime(daily_data['Date'], utc=True).dt.tz_localize(None)
                daily_data.set_index('Date', inplace=True)
            
            # Aggregate to weekly
            weekly_data = weekly_aggregator.aggregate(daily_data)
            
            # Create features
            features_data = create_simple_features(weekly_data)
            
            # Create targets
            targets_data = create_simple_targets(features_data, FORECAST_HORIZON)
            
            backtesting_data[symbol] = targets_data
            print(f"   ✅ {symbol}: {len(targets_data)} weeks, {len(targets_data.columns)} features")
            
        except Exception as e:
            print(f"   ❌ {symbol}: Error - {e}")

print(f"\n✅ Data preparation complete for {len(backtesting_data)} stocks")


## 4. Quarterly Backtesting Execution


In [ ]:
# Perform quarterly backtesting for all stocks
print("🔄 Starting quarterly backtesting...")

backtesting_results = {}
overall_performance = {}

for symbol in STOCK_SYMBOLS:
    if symbol in backtesting_data:
        print(f"\n📈 Backtesting {symbol}...")
        
        try:
            # Prepare data
            data = backtesting_data[symbol].dropna()
            
            if len(data) < 20:  # Need minimum data
                print(f"   ⚠️ {symbol}: Insufficient data ({len(data)} rows)")
                continue
            
            # Create standalone backtester with quarterly parameters
            backtester = StandaloneBacktester(
                initial_train_size=QUARTERLY_PARAMS['initial_train_size'],
                test_size=QUARTERLY_PARAMS['test_size'],
                step_size=QUARTERLY_PARAMS['step_size'],
                min_train_size=QUARTERLY_PARAMS['min_train_size'],
                target_column=QUARTERLY_PARAMS['target_column'],
                forecast_horizon=QUARTERLY_PARAMS['forecast_horizon'],
                model_params=OPTIMIZED_HYPERPARAMS,
                random_state=42,
                verbose=True
            )
            
            # Run backtesting
            results = backtester.backtest(data)
            
            # Store results
            backtesting_results[symbol] = {
                'backtester': backtester,
                'results': results,
                'data_points': len(data),
                'windows': len(backtester.results)
            }
            
            # Extract performance metrics
            overall_performance[symbol] = {
                'mape': results['overall_mape'],
                'rmse': results['overall_rmse'],
                'r2': results['overall_r2'],
                'windows': len(backtester.results)
            }
            
            print(f"   ✅ {symbol}: MAPE={results['overall_mape']:.2f}%, RMSE={results['overall_rmse']:.2f}, R²={results['overall_r2']:.3f}")
            
        except Exception as e:
            print(f"   ❌ {symbol}: Backtesting failed - {e}")
            import traceback
            traceback.print_exc()

print(f"\n✅ Backtesting complete for {len(backtesting_results)} stocks")


## 5. Performance Analysis and Results


In [ ]:
# Display comprehensive performance results
print("=" * 80)
print("📊 QUARTERLY BACKTESTING RESULTS")
print("=" * 80)

print(f"\n🎯 Backtesting Summary:")
print(f"   • Stocks Backtested: {len(backtesting_results)}")
print(f"   • Forecast Horizon: {FORECAST_HORIZON} weeks")
print(f"   • Validation Windows: End of each quarter")
print(f"   • Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📈 Performance Metrics:")
print(f"{'Stock':<8} {'MAPE (%)':<10} {'RMSE':<10} {'R²':<8} {'Windows':<8} {'Data Points':<12}")
print("-" * 70)

for symbol in STOCK_SYMBOLS:
    if symbol in overall_performance:
        perf = overall_performance[symbol]
        print(f"{symbol:<8} {perf['mape']:<10.2f} {perf['rmse']:<10.2f} {perf['r2']:<8.3f} {perf['windows']:<8} {backtesting_results[symbol]['data_points']:<12}")
    else:
        print(f"{symbol:<8} {'N/A':<10} {'N/A':<10} {'N/A':<8} {'N/A':<8} {'N/A':<12}")

# Calculate portfolio-level statistics
if overall_performance:
    mape_values = [perf['mape'] for perf in overall_performance.values()]
    rmse_values = [perf['rmse'] for perf in overall_performance.values()]
    r2_values = [perf['r2'] for perf in overall_performance.values()]
    
    print(f"\n📊 Portfolio Statistics:")
    print(f"   • Average MAPE: {np.mean(mape_values):.2f}%")
    print(f"   • Average RMSE: {np.mean(rmse_values):.2f}")
    print(f"   • Average R²: {np.mean(r2_values):.3f}")
    print(f"   • Best MAPE: {min(mape_values):.2f}% ({min(overall_performance.keys(), key=lambda x: overall_performance[x]['mape'])})")
    print(f"   • Best R²: {max(r2_values):.3f} ({max(overall_performance.keys(), key=lambda x: overall_performance[x]['r2'])})")

print(f"\n🔧 Model Configuration:")
print(f"   • Hyperparameters: Optimized from forecasting notebook")
print(f"   • Training Strategy: Quarterly rolling windows")
print(f"   • Feature Engineering: Simple features (lags, rolling, time)")
print(f"   • Target: {FORECAST_HORIZON}-week percentage change")

print("\n" + "=" * 80)
print("✅ Quarterly backtesting analysis completed!")
print("=" * 80)


## 6. Visualization


In [ ]:
# Create comprehensive visualization of backtesting results
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Quarterly Backtesting Results Analysis', fontsize=20, fontweight='bold')

# 1. Performance metrics comparison
ax1 = axes[0, 0]
if overall_performance:
    symbols = list(overall_performance.keys())
    mape_values = [overall_performance[s]['mape'] for s in symbols]
    colors = plt.cm.Set1(np.linspace(0, 1, len(symbols)))
    
    bars = ax1.bar(symbols, mape_values, color=colors, alpha=0.7)
    ax1.set_title('MAPE Performance by Stock', fontsize=14, fontweight='bold')
    ax1.set_ylabel('MAPE (%)')
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, mape_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{value:.1f}%', ha='center', va='bottom')

# 2. R² scores comparison
ax2 = axes[0, 1]
if overall_performance:
    r2_values = [overall_performance[s]['r2'] for s in symbols]
    
    bars = ax2.bar(symbols, r2_values, color=colors, alpha=0.7)
    ax2.set_title('R² Performance by Stock', fontsize=14, fontweight='bold')
    ax2.set_ylabel('R² Score')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, r2_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{value:.3f}', ha='center', va='bottom')

# 3. Number of validation windows
ax3 = axes[1, 0]
if overall_performance:
    windows = [overall_performance[s]['windows'] for s in symbols]
    
    bars = ax3.bar(symbols, windows, color=colors, alpha=0.7)
    ax3.set_title('Number of Validation Windows', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Number of Windows')
    ax3.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, windows):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{value}', ha='center', va='bottom')

# 4. Performance scatter plot (MAPE vs R²)
ax4 = axes[1, 1]
if overall_performance:
    scatter = ax4.scatter(mape_values, r2_values, c=range(len(symbols)), 
                         cmap='Set1', s=200, alpha=0.7)
    
    # Add stock labels
    for i, symbol in enumerate(symbols):
        ax4.annotate(symbol, (mape_values[i], r2_values[i]), 
                    xytext=(5, 5), textcoords='offset points', fontweight='bold')
    
    ax4.set_title('MAPE vs R² Performance', fontsize=14, fontweight='bold')
    ax4.set_xlabel('MAPE (%)')
    ax4.set_ylabel('R² Score')
    ax4.grid(True, alpha=0.3)
    
    # Add quadrant lines
    ax4.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='R² = 0.5')
    ax4.axvline(x=np.mean(mape_values), color='blue', linestyle='--', alpha=0.5, label='Avg MAPE')
    ax4.legend()

plt.tight_layout()
plt.show()


## 7. Results Export and Summary


In [ ]:
# Export backtesting results
import json
from datetime import datetime

# Create results summary
results_summary = {
    'backtesting_date': datetime.now().isoformat(),
    'configuration': {
        'stock_symbols': STOCK_SYMBOLS,
        'forecast_horizon': FORECAST_HORIZON,
        'quarterly_params': QUARTERLY_PARAMS,
        'hyperparameters': OPTIMIZED_HYPERPARAMS
    },
    'performance': overall_performance,
    'summary_statistics': {}
}

# Add summary statistics
if overall_performance:
    mape_values = [perf['mape'] for perf in overall_performance.values()]
    rmse_values = [perf['rmse'] for perf in overall_performance.values()]
    r2_values = [perf['r2'] for perf in overall_performance.values()]
    
    results_summary['summary_statistics'] = {
        'average_mape': float(np.mean(mape_values)),
        'average_rmse': float(np.mean(rmse_values)),
        'average_r2': float(np.mean(r2_values)),
        'best_mape': float(min(mape_values)),
        'best_r2': float(max(r2_values)),
        'worst_mape': float(max(mape_values)),
        'worst_r2': float(min(r2_values)),
        'total_stocks': len(overall_performance)
    }

# Save results to file
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_filename = f'backtesting_results_{timestamp}.json'

with open(results_filename, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"📁 Results exported to: {results_filename}")

# Display final summary
print("\n" + "=" * 80)
print("🎯 QUARTERLY BACKTESTING SUMMARY")
print("=" * 80)

print(f"\n📊 Key Findings:")
if overall_performance:
    best_stock = min(overall_performance.keys(), key=lambda x: overall_performance[x]['mape'])
    worst_stock = max(overall_performance.keys(), key=lambda x: overall_performance[x]['mape'])
    
    print(f"   • Best Performing Stock: {best_stock} (MAPE: {overall_performance[best_stock]['mape']:.2f}%)")
    print(f"   • Most Challenging Stock: {worst_stock} (MAPE: {overall_performance[worst_stock]['mape']:.2f}%)")
    print(f"   • Portfolio Average MAPE: {np.mean(mape_values):.2f}%")
    print(f"   • Portfolio Average R²: {np.mean(r2_values):.3f}")

print(f"\n🔧 Model Performance:")
print(f"   • Hyperparameters: Optimized from forecasting notebook")
print(f"   • Validation Strategy: Quarterly rolling windows")
print(f"   • Feature Engineering: Simple, robust features")
print(f"   • Target: {FORECAST_HORIZON}-week percentage change")

print(f"\n📈 Recommendations:")
if overall_performance:
    avg_mape = np.mean(mape_values)
    if avg_mape < 5:
        print(f"   ✅ Excellent performance (MAPE < 5%)")
    elif avg_mape < 10:
        print(f"   ✅ Good performance (MAPE < 10%)")
    elif avg_mape < 15:
        print(f"   ⚠️ Moderate performance (MAPE < 15%)")
    else:
        print(f"   ❌ Poor performance (MAPE > 15%)")
    
    avg_r2 = np.mean(r2_values)
    if avg_r2 > 0.7:
        print(f"   ✅ Strong predictive power (R² > 0.7)")
    elif avg_r2 > 0.5:
        print(f"   ✅ Moderate predictive power (R² > 0.5)")
    else:
        print(f"   ⚠️ Weak predictive power (R² < 0.5)")

print(f"\n📋 Next Steps:")
print(f"   • Review individual stock performance patterns")
print(f"   • Consider feature engineering improvements for underperforming stocks")
print(f"   • Validate results with out-of-sample testing")
print(f"   • Monitor model performance over time")

print("\n" + "=" * 80)
print("✅ Quarterly backtesting completed successfully!")
print("=" * 80)


## 8. Usage Instructions

### **How to Use This Backtesting Notebook:**

1. **Run All Cells**: Execute all cells to perform complete quarterly backtesting
2. **Customize Stocks**: Modify `STOCK_SYMBOLS` to test different stocks
3. **Adjust Parameters**: Change `FORECAST_HORIZON` or quarterly parameters as needed
4. **Review Results**: Analyze performance metrics and visualizations

### **Quarterly Validation Strategy:**

- **✅ End-of-Quarter Windows**: Validation windows end at the end of each quarter
- **🔄 Rolling Training**: Training data expands with each new quarter
- **📊 Consistent Evaluation**: Each quarter provides one validation point
- **⏰ Realistic Timeline**: Mimics real-world quarterly reporting cycles

### **Hyperparameter Integration:**

- **🎯 Optimized Parameters**: Uses hyperparameters from stock forecasting notebook
- **🔧 No Tuning**: Skips hyperparameter tuning for faster execution
- **📈 Proven Performance**: Leverages previously optimized model settings
- **⚡ Efficient**: Focuses on validation rather than optimization

### **Performance Metrics:**

- **MAPE**: Mean Absolute Percentage Error (primary metric)
- **RMSE**: Root Mean Square Error
- **R²**: Coefficient of determination
- **Windows**: Number of validation periods

### **Results Export:**

- **📁 JSON Export**: Detailed results saved to timestamped file
- **📊 Visualizations**: Comprehensive charts and analysis
- **📋 Summary Report**: Key findings and recommendations
- **🔍 Individual Analysis**: Stock-by-stock performance breakdown

### **Benefits:**

- ⚡ **Fast Execution**: No hyperparameter tuning required
- 📊 **Comprehensive**: Multi-stock analysis with detailed metrics
- 🎯 **Realistic**: Quarterly validation matches business cycles
- 📈 **Actionable**: Clear performance insights and recommendations
- 🔄 **Reproducible**: Consistent methodology across all stocks
